# Notebook 3 — `training_sum.ipynb`

**Sovereign Dialect-Bridge · Step 3 — Train Three Summarizers (Stage 2)**

Fine-tune tiga model summarizer pada IndoSum bersih (10K samples). Model:

| Name | Checkpoint | Type | Prefix? |
|------|------------|------|---------|
| IndoT5   | `cahya/t5-base-indonesian-summarization-cased` | t5   | tidak |
| IndoBART | `indobenchmark/indoBART`                       | bart | tidak |
| mT5-base | `google/mt5-base`                              | mt5  | **WAJIB `"summarize: "`** |

Train berurutan: load → train → eval quick → `free_vram()` → model berikutnya. Total ~3.75 jam di RTX 3090.


## 1. Setup environment


In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"]   = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import gc, re, json, random, time
from pathlib import Path

import numpy as np
import pandas as pd
import torch

DEVICE   = "cuda" if torch.cuda.is_available() else "cpu"
USE_BF16 = torch.cuda.is_bf16_supported() if DEVICE == "cuda" else False
USE_FP16 = False

assert DEVICE == "cuda", "Notebook ini butuh GPU."
print(f"GPU  : {torch.cuda.get_device_name(0)}")
print(f"VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB  |  bf16: {USE_BF16}")


## 2. Install dependencies

```bash
pip install transformers==4.40.0 datasets accelerate sentencepiece PySastrawi rouge-score sumy networkx
```


## 3. Configuration & paths


In [ ]:
CWD = Path.cwd()
ROOT = CWD if (CWD / "data").exists() else CWD.parent
DATA_DIR   = ROOT / "data"
MODELS_DIR = ROOT / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

# Sinkron dengan CLAUDE.md
INDOT5_MODEL   = "cahya/t5-base-indonesian-summarization-cased"
INDOBART_MODEL = "indobenchmark/indoBART"
MT5BASE_MODEL  = "google/mt5-base"

SUMM_MAX_INPUT     = 512
SUMM_MAX_TARGET    = 150
SUMM_LR            = 5e-5
SUMM_EPOCHS        = 3
SUMM_BATCH         = 8
SUMM_GRAD_ACCUM    = 2
SUMM_MAX_GRAD_NORM = 1.0
RANDOM_SEED        = 42

GEN_KWARGS = dict(
    max_new_tokens       = SUMM_MAX_TARGET,
    num_beams            = 4,
    no_repeat_ngram_size = 3,
    early_stopping       = True,
    length_penalty       = 1.0,
)

random.seed(RANDOM_SEED); np.random.seed(RANDOM_SEED); torch.manual_seed(RANDOM_SEED)
print(f"ROOT       : {ROOT}")
print(f"effective batch = {SUMM_BATCH * SUMM_GRAD_ACCUM}")


## 4. Load IndoSum parquet


In [ ]:
df_train = pd.read_parquet(DATA_DIR / "train.parquet")
df_val   = pd.read_parquet(DATA_DIR / "val.parquet")
df_test  = pd.read_parquet(DATA_DIR / "test.parquet")

print(f"Train: {len(df_train):,}  |  Val: {len(df_val):,}  |  Test: {len(df_test):,}")
print(f"Avg word_count        : {df_train['word_count'].mean():.0f}")
print(f"Avg summary_word_count: {df_train['summary_word_count'].mean():.0f}")
print(f"Avg compression_ratio : {df_train['compression_ratio'].mean():.4f}")


## 5. Preprocessing — DUA jalur TERPISAH

> **Kritis:** `preprocess_abstractive()` TIDAK boleh memanggil `stem_text()`. Model neural dilatih pada teks unstemmed; stemming menyebabkan distributional shift dan menurunkan ROUGE 3–8 poin.


In [ ]:
try:
    from PySastrawi.Stemmer.StemmerFactory import StemmerFactory   # versi lama
except ImportError:
    from Sastrawi.Stemmer.StemmerFactory import StemmerFactory     # versi pip terbaru

_stemmer = StemmerFactory().create_stemmer()

STEM_STOPWORDS = {
    "rt","rw","jl","bpbd","dinas","kelurahan","kecamatan","lapor",
    "laporan","pengaduan","warga","masyarakat","pemerintah","kantor",
    "jalan","gedung","sekolah","puskesmas","pasar","sungai","jembatan",
}


def clean_noise(text: str) -> str:
    text = re.sub(r"https?://\S+", "", text)
    text = re.sub(r"(.)\1{2,}", r"\1\1", text)
    return re.sub(r"\s+", " ", text).strip()


def normalize_case_punct(text: str) -> str:
    text = text.lower()
    text = re.sub(r"[!?]{2,}", "!", text)
    return re.sub(r"\s([.,!?;:])", r"\1", text).strip()


def stem_text(text: str) -> str:
    return " ".join(
        w if w.lower() in STEM_STOPWORDS else _stemmer.stem(w)
        for w in text.split()
    )


def preprocess_extractive(text: str) -> str:
    """Untuk TextRank / NER — DENGAN stemming."""
    text = clean_noise(text)
    text = normalize_case_punct(text)
    text = stem_text(text)
    return text


def preprocess_abstractive(text: str) -> str:
    """Untuk IndoT5 / IndoBART / mT5-base — TANPA stemming."""
    text = clean_noise(text)
    text = normalize_case_punct(text)
    return text   # STOP — TIDAK ADA stem_text() di sini


# Sanity check
demo = "Saya melaporkan jalan rusak di RT 05!! https://contoh.id/x"
print("RAW :", demo)
print("EXT :", preprocess_extractive(demo))
print("ABS :", preprocess_abstractive(demo))


## 6. Model registry — daftar model yang ditraining


In [ ]:
MODELS_TO_TRAIN = [
    {"name": "IndoT5",   "model_id": INDOT5_MODEL,   "model_type": "t5",
     "output_dir": str(MODELS_DIR / "indot5")},
    {"name": "IndoBART", "model_id": INDOBART_MODEL, "model_type": "bart",
     "output_dir": str(MODELS_DIR / "indobart")},
    {"name": "mT5-base", "model_id": MT5BASE_MODEL,  "model_type": "mt5",
     "output_dir": str(MODELS_DIR / "mt5base")},
]

for spec in MODELS_TO_TRAIN:
    Path(spec["output_dir"]).mkdir(parents=True, exist_ok=True)

print("Akan training:")
for s in MODELS_TO_TRAIN:
    print(f"  {s['name']:10s} → {s['output_dir']}")


## 7. Tokenization helper

Apply `preprocess_abstractive` ke input dan tambahkan prefix `"summarize: "` HANYA untuk mT5. Summary TIDAK di-preprocess (biarkan natural untuk ROUGE).


In [ ]:
def build_hf_dataset(df, tokenizer, model_type, max_input=SUMM_MAX_INPUT, max_target=SUMM_MAX_TARGET):
    from datasets import Dataset

    inputs  = df["text"].apply(preprocess_abstractive).tolist()
    if model_type == "mt5":
        inputs = ["summarize: " + t for t in inputs]
    targets = df["summary"].astype(str).tolist()

    def tok_fn(batch):
        model_inputs = tokenizer(batch["input"],  max_length=max_input,  truncation=True)
        labels       = tokenizer(text_target=batch["target"], max_length=max_target, truncation=True)
        model_inputs["labels"] = [
            [(tok if tok != tokenizer.pad_token_id else -100) for tok in seq]
            for seq in labels["input_ids"]
        ]
        return model_inputs

    ds = Dataset.from_dict({"input": inputs, "target": targets})
    return ds.map(tok_fn, batched=True, remove_columns=ds.column_names)


## 8. Training function — `train_one_model`


In [ ]:
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments, Seq2SeqTrainer,
)


def free_vram(*objs):
    for o in objs:
        del o
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def quick_rouge(model, tokenizer, df, model_type, n=50):
    """Quick ROUGE check on n samples for sanity."""
    from rouge_score import rouge_scorer
    model.eval()
    scorer = rouge_scorer.RougeScorer(["rouge1","rouge2","rougeL"], use_stemmer=False)
    rouge_sums = {"rouge1":0.0, "rouge2":0.0, "rougeL":0.0}
    sample = df.sample(min(n, len(df)), random_state=RANDOM_SEED)
    for _, row in sample.iterrows():
        text = preprocess_abstractive(row["text"])
        if model_type == "mt5":
            text = "summarize: " + text
        inputs = tokenizer(text, return_tensors="pt", truncation=True,
                           max_length=SUMM_MAX_INPUT).to(DEVICE)
        with torch.no_grad():
            out = model.generate(**inputs, **GEN_KWARGS)
        pred = tokenizer.decode(out[0], skip_special_tokens=True)
        sc = scorer.score(row["summary"], pred)
        for k in rouge_sums:
            rouge_sums[k] += sc[k].fmeasure
    n_eff = len(sample)
    return {k: round(v / n_eff, 4) for k, v in rouge_sums.items()}


def train_one_model(spec, df_train, df_val):
    print(f"\n{'='*60}")
    print(f"Training {spec['name']}")
    print(f"{'='*60}")
    t0 = time.time()

    tokenizer = AutoTokenizer.from_pretrained(spec["model_id"], use_fast=False if spec["model_type"]=="bart" else True)
    model     = AutoModelForSeq2SeqLM.from_pretrained(spec["model_id"])

    if model.config.decoder_start_token_id is None and tokenizer.pad_token_id is not None:
        model.config.decoder_start_token_id = tokenizer.pad_token_id

    ds_train_tok = build_hf_dataset(df_train, tokenizer, spec["model_type"])
    ds_val_tok   = build_hf_dataset(df_val,   tokenizer, spec["model_type"])
    print(f"  tokenized train: {len(ds_train_tok):,}  |  val: {len(ds_val_tok):,}")

    collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model,
                                      padding="longest", return_tensors="pt")

    args = Seq2SeqTrainingArguments(
        output_dir                  = spec["output_dir"],
        num_train_epochs            = SUMM_EPOCHS,
        per_device_train_batch_size = SUMM_BATCH,
        per_device_eval_batch_size  = SUMM_BATCH,
        gradient_accumulation_steps = SUMM_GRAD_ACCUM,
        learning_rate               = SUMM_LR,
        warmup_ratio                = 0.1,
        weight_decay                = 0.01,
        max_grad_norm               = SUMM_MAX_GRAD_NORM,
        bf16                        = USE_BF16,
        fp16                        = False,
        predict_with_generate       = True,
        generation_max_length       = SUMM_MAX_TARGET,
        generation_num_beams        = 4,
        eval_strategy               = "epoch",
        save_strategy               = "epoch",
        load_best_model_at_end      = True,
        metric_for_best_model       = "eval_loss",
        greater_is_better           = False,
        save_total_limit            = 2,
        label_smoothing_factor      = 0.1,
        gradient_checkpointing      = True,
        logging_steps               = 50,
        report_to                   = "none",
        seed                        = RANDOM_SEED,
    )

    trainer = Seq2SeqTrainer(
        model=model, args=args, train_dataset=ds_train_tok, eval_dataset=ds_val_tok,
        data_collator=collator, tokenizer=tokenizer,
    )

    trainer.train()
    trainer.save_model(spec["output_dir"])
    tokenizer.save_pretrained(spec["output_dir"])

    model.to(DEVICE)
    quick = quick_rouge(model, tokenizer, df_val, spec["model_type"], n=50)
    elapsed = (time.time() - t0) / 60
    print(f"  Quick ROUGE on val (n=50): {quick}")
    print(f"  Elapsed: {elapsed:.1f} min")

    # Save metadata
    meta = {"name": spec["name"], "model_id": spec["model_id"],
            "model_type": spec["model_type"], "quick_rouge": quick,
            "elapsed_min": round(elapsed, 2)}
    with open(Path(spec["output_dir"]) / "training_meta.json", "w") as f:
        json.dump(meta, f, indent=2)

    free_vram(model, trainer, tokenizer)
    return meta


## 9. Train semua model berurutan


In [ ]:
all_meta = []
for spec in MODELS_TO_TRAIN:
    meta = train_one_model(spec, df_train, df_val)
    all_meta.append(meta)

print("\n" + "="*60)
print("Training summary:")
for m in all_meta:
    print(f"  {m['name']:10s}  R1={m['quick_rouge']['rouge1']:.4f}  "
          f"R2={m['quick_rouge']['rouge2']:.4f}  RL={m['quick_rouge']['rougeL']:.4f}  "
          f"({m['elapsed_min']:.1f} min)")


## 10. TextRank baseline (untuk perbandingan di Notebook 4)

> **Kritis:** `max_words=80` cutoff WAJIB ada. Tanpa ini CR baseline bisa mencapai 0.985 → ROUGE jadi tidak bermakna.


In [ ]:
import networkx as nx
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


def split_sentences(text: str) -> list[str]:
    sents = re.split(r"(?<=[.!?])\s+", text.strip())
    return [s.strip() for s in sents if s.strip()]


def summarize_textrank(text: str, n_sentences: int = 3, max_words: int = 80) -> str:
    """TextRank dengan hard cap max_words=80 (CR safe)."""
    sents = split_sentences(preprocess_extractive(text))
    if len(sents) <= n_sentences:
        result = " ".join(sents)
    else:
        try:
            vec = TfidfVectorizer().fit_transform(sents)
            sim = cosine_similarity(vec)
            np.fill_diagonal(sim, 0)
            g = nx.from_numpy_array(sim)
            scores = nx.pagerank(g, max_iter=200)
            ranked = sorted(range(len(sents)), key=lambda i: scores[i], reverse=True)
            selected = sorted(ranked[:n_sentences])
            result = " ".join(sents[i] for i in selected)
        except Exception:
            result = " ".join(sents[:n_sentences])

    words = result.split()
    if len(words) > max_words:
        result = " ".join(words[:max_words])
    return result


# Sanity check
sample_text = df_train.iloc[0]["text"]
tr_summary = summarize_textrank(sample_text, n_sentences=3, max_words=80)
print(f"TextRank sanity check:")
print(f"  input  : {sample_text[:120]}...")
print(f"  output ({len(tr_summary.split())} words): {tr_summary}")


## 11. NER baseline (extractive berdasar named entities)


In [ ]:
def summarize_ner(text: str, n_sentences: int = 3, max_words: int = 80) -> str:
    """Pilih kalimat dengan density kata berkapital terbanyak (proxy named entities)."""
    raw_sents = split_sentences(text)
    if not raw_sents:
        return ""

    def entity_density(s: str) -> float:
        words = s.split()
        if not words:
            return 0.0
        caps = sum(1 for w in words[1:] if w and w[0].isupper())   # exclude first word
        return caps / len(words)

    scores = [(i, entity_density(s)) for i, s in enumerate(raw_sents)]
    ranked = sorted(scores, key=lambda x: x[1], reverse=True)[:n_sentences]
    selected = sorted(i for i, _ in ranked)
    result = " ".join(raw_sents[i] for i in selected)

    words = result.split()
    if len(words) > max_words:
        result = " ".join(words[:max_words])
    return result


ner_summary = summarize_ner(sample_text)
print(f"NER baseline sanity check:")
print(f"  output ({len(ner_summary.split())} words): {ner_summary}")


## 12. Save helper functions untuk Notebook 4

Notebook 4 (inference) akan re-import fungsi-fungsi ini. Simpan ke file Python helper agar tidak duplikasi.


In [ ]:
HELPERS_PATH = ROOT / "notebook" / "_helpers.py"
HELPERS_CONTENT = '''"""Shared helpers for Sovereign Dialect-Bridge inference notebook.

Diekspor dari training_sum.ipynb. Jangan edit langsung — update notebook lalu re-export.
"""
import re
import numpy as np
import networkx as nx
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

try:
    from PySastrawi.Stemmer.StemmerFactory import StemmerFactory
except ImportError:
    from Sastrawi.Stemmer.StemmerFactory import StemmerFactory

_stemmer = StemmerFactory().create_stemmer()

STEM_STOPWORDS = {
    "rt","rw","jl","bpbd","dinas","kelurahan","kecamatan","lapor",
    "laporan","pengaduan","warga","masyarakat","pemerintah","kantor",
    "jalan","gedung","sekolah","puskesmas","pasar","sungai","jembatan",
}


def clean_noise(text):
    text = re.sub(r"https?://\\S+", "", text)
    text = re.sub(r"(.)\\1{2,}", r"\\1\\1", text)
    return re.sub(r"\\s+", " ", text).strip()


def normalize_case_punct(text):
    text = text.lower()
    text = re.sub(r"[!?]{2,}", "!", text)
    return re.sub(r"\\s([.,!?;:])", r"\\1", text).strip()


def stem_text(text):
    return " ".join(
        w if w.lower() in STEM_STOPWORDS else _stemmer.stem(w)
        for w in text.split()
    )


def preprocess_extractive(text):
    return stem_text(normalize_case_punct(clean_noise(text)))


def preprocess_abstractive(text):
    return normalize_case_punct(clean_noise(text))


def split_sentences(text):
    sents = re.split(r"(?<=[.!?])\\s+", text.strip())
    return [s.strip() for s in sents if s.strip()]


def summarize_textrank(text, n_sentences=3, max_words=80):
    sents = split_sentences(preprocess_extractive(text))
    if len(sents) <= n_sentences:
        result = " ".join(sents)
    else:
        try:
            vec = TfidfVectorizer().fit_transform(sents)
            sim = cosine_similarity(vec)
            np.fill_diagonal(sim, 0)
            g = nx.from_numpy_array(sim)
            scores = nx.pagerank(g, max_iter=200)
            ranked = sorted(range(len(sents)), key=lambda i: scores[i], reverse=True)
            selected = sorted(ranked[:n_sentences])
            result = " ".join(sents[i] for i in selected)
        except Exception:
            result = " ".join(sents[:n_sentences])
    words = result.split()
    if len(words) > max_words:
        result = " ".join(words[:max_words])
    return result


def summarize_ner(text, n_sentences=3, max_words=80):
    raw_sents = split_sentences(text)
    if not raw_sents:
        return ""
    def density(s):
        words = s.split()
        if not words:
            return 0.0
        return sum(1 for w in words[1:] if w and w[0].isupper()) / len(words)
    scores = [(i, density(s)) for i, s in enumerate(raw_sents)]
    ranked = sorted(scores, key=lambda x: x[1], reverse=True)[:n_sentences]
    selected = sorted(i for i, _ in ranked)
    result = " ".join(raw_sents[i] for i in selected)
    words = result.split()
    if len(words) > max_words:
        result = " ".join(words[:max_words])
    return result


GEN_KWARGS = dict(
    max_new_tokens=150, num_beams=4, no_repeat_ngram_size=3,
    early_stopping=True, length_penalty=1.0,
)
'''

with open(HELPERS_PATH, "w", encoding="utf-8") as f:
    f.write(HELPERS_CONTENT)
print(f"Saved helpers → {HELPERS_PATH}")


## ✅ Selesai

Output:
- `models/indot5/`, `models/indobart/`, `models/mt5base/` — checkpoint summarizer + tokenizer + `training_meta.json`
- `notebook/_helpers.py` — fungsi preprocessing & baseline yang dipakai Notebook 4

**Langkah selanjutnya:** jalankan `inference.ipynb` untuk evaluasi lengkap + demo end-to-end.
